In [1]:
import sys 

sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import torch.backends.cudnn as cudnn
import random
from mamba_ssm import Mamba

from thop import clever_format
from ptflops import get_model_complexity_info

import os
os.chdir("/workspace/dehazing")

In [2]:
def set_seed(seed):
    """Sets the seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        cudnn.deterministic = True
        cudnn.benchmark = False

## Stage 1 Code

In [3]:
def get_pad_layer(pad_type):
    if(pad_type in ['refl','reflect']):
        PadLayer = nn.ReflectionPad2d
    elif(pad_type in ['repl','replicate']):
        PadLayer = nn.ReplicationPad2da
    elif(pad_type=='zero'):
        PadLayer = nn.ZeroPad2d
    else:
        print(f'Pad type [{pad_type}] not recognized')
    return PadLayer


class AntiAlias_Downsample(nn.Module):
    def __init__(self, channels, pad_type = 'reflect', filt_size = 3, 
                        stride = 2, pad_off = 0):
        super(AntiAlias_Downsample, self).__init__()
        self.filt_size = filt_size
        self.pad_off = pad_off
        self.pad_type = pad_type

        # Asymmetric padding (round up at the top and round down at the bottom)
        # Perfect when kernel size is 2 
        self.pad_sizes = [int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2)),
                          int(1. * (filt_size - 1) / 2), int(np.ceil(1. * (filt_size - 1) / 2))]
        self.pad_sizes = [pad_size + pad_off for pad_size in self.pad_sizes]
        self.stride = stride 
        self.off = int((self.stride - 1) / 2.)
        self.channels = channels 

        # Define the binomial filter weights
        if(self.filt_size==1):
            a = np.array([1.,])
        elif(self.filt_size==2):
            a = np.array([1., 1.])
        elif(self.filt_size==3):
            a = np.array([1., 2., 1.])
        elif(self.filt_size==4):    
            a = np.array([1., 3., 3., 1.])
        elif(self.filt_size==5):    
            a = np.array([1., 4., 6., 4., 1.])
        elif(self.filt_size==6):    
            a = np.array([1., 5., 10., 10., 5., 1.])
        elif(self.filt_size==7):    
            a = np.array([1., 6., 15., 20., 15., 6., 1.])
            
        # Create a 2D filter by taking the outer product of the 1D filter
        filt = torch.tensor(a[:, None] * a[None, :], dtype = torch.float32)
        filt = filt / torch.sum(filt) # Normalize

        # Reshape to (out_channels, in_channels/groups, kH, kW) for 
        # depthwise convolution
        filt = filt.view(1, 1, filt_size, filt_size)
        filt = filt.repeat(channels, 1, 1, 1)

        # Register as a buffer so PyTorch knows these are NOT trainable parameters
        self.register_buffer('filt', filt)
        self.pad = get_pad_layer(pad_type)(self.pad_sizes)

    def forward(self, inp):
        if (self.filt_size == 1):
            if (self.pad_off == 0):
                return inp[:, :, ::self.stride, ::self.stride] 
            else:
                return self.pad(inp)[:, :, ::self.stride, ::self.stride] 

        else:
            return F.conv2d(self.pad(inp), self.filt, stride = self.stride, groups = inp.shape[1])


class VariantB_AntiAliasedDownsample(nn.Module):
    """
    Anti-Aliased Downsampling (BlurPool) based on Richard Zhang's paper.
    Preserves shift-invariance and prevents high-frequency aliasing.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # 1. Feature Mixing (Stride 1 preserves all spatial information)
        self.conv = nn.Conv2d(dim_in, dim_out, kernel_size=3, stride=1, padding=1)
        # 2. Anti-aliased spatial reduction (Low-pass filter + subsampling)
        # NOTE: Make sure your AntiAlias_Downsample class is defined in the script!
        self.aa_down = AntiAlias_Downsample(channels=dim_out, filt_size=3, stride=2)

    def forward(self, x):
        return self.aa_down(self.conv(x))

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Stage1_DCP_Prior(nn.Module):
    """
    Refactored for Structural Sharpness:
    - Replaces Dilation with Depthwise Separable Bottleneck (Preserves local edges).
    - Swaps Bilinear Upsampling for PixelShuffle (Prevents interpolation blur).
    - Uses a Gated Residual connection for the Transmission map.
    """
    def __init__(self, in_channels=4, base_dim=32):
        super().__init__()
        self.variant = 'S_Sharp' # 'S' for Structural Sharpness
        
        # 1. Initial Encoder (RGB + DCP)
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)
        self.enc1 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)

        # 2. Downsampling (Anti-Aliased)
        self.down1 = VariantB_AntiAliasedDownsample(base_dim, base_dim * 2)
        self.down2 = VariantB_AntiAliasedDownsample(base_dim * 2, base_dim * 4)

        # 3. Dilated Bottleneck (1 -> 2 -> 1 Dilation Pattern)
        # This expands the receptive field to capture "haze context" 
        # while maintaining local edge precision.
        self.bottleneck = nn.Sequential(
            # Dilation 1: Local context
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Dilation 2: Mid-range context (Captures haze gradients)
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Dilation 1: Refine back to local structural details
            nn.Conv2d(base_dim * 4, base_dim * 4, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm2d(base_dim * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )        
        
        # 4. Sharp Decoder (Using PixelShuffle to avoid interpolation blur)
        # Note: PixelShuffle(upscale_factor=2) reduces channels by 4x
        self.up1_ps = nn.Sequential(
            nn.Conv2d(base_dim * 4, base_dim * 8, kernel_size=1),
            nn.PixelShuffle(2)
        ) # Result: base_dim * 2 channels
        
        self.dec1_fusion = nn.Conv2d(base_dim * 4, base_dim * 2, kernel_size=1) 
        self.dec1 = nn.Conv2d(base_dim * 2, base_dim * 2, kernel_size=3, padding=1)
        
        self.up2_ps = nn.Sequential(
            nn.Conv2d(base_dim * 2, base_dim * 4, kernel_size=1),
            nn.PixelShuffle(2)
        ) # Result: base_dim channels
        
        self.dec2_fusion = nn.Conv2d(base_dim * 2, base_dim, kernel_size=1)
        self.dec2 = nn.Conv2d(base_dim, base_dim, kernel_size=3, padding=1)
        
        # 5. Output Heads
        self.t_head_residual = nn.Sequential(
            nn.Conv2d(base_dim, 1, kernel_size=3, padding=1), 
            nn.Tanh() 
        )

        self.A_head_spatial = nn.Sequential(
            nn.Conv2d(base_dim, 3, kernel_size=3, padding=1), 
            nn.Sigmoid() 
        )

        # self.A_head_global = nn.Sequential(
        #     nn.AdaptiveAvgPool2d(1),
        #     nn.Flatten(),
        #     nn.Linear(base_dim, base_dim // 2),
        #     nn.ReLU(inplace=True),
        #     nn.Linear(base_dim // 2, 3),
        #     nn.Sigmoid()
        # )

    def forward(self, hazy_img, t_dcp):
        x = torch.cat([hazy_img, t_dcp], dim=1)

        # Encoder
        x = F.relu(self.init_conv(x))
        e1 = F.relu(self.enc1(x))
        d1 = self.down1(e1)
        e2 = F.relu(self.enc2(d1))
        d2 = self.down2(e2)

        # Bottleneck (Preserves structural edges)
        b = self.bottleneck(d2)
        
        # Decoder 1: PixelShuffle + Skip Connection
        u1 = torch.cat([self.up1_ps(b), e2], dim=1)
        u1 = F.relu(self.dec1_fusion(u1))
        u1 = F.relu(self.dec1(u1))
        
        # Decoder 2: PixelShuffle + Skip Connection
        u2 = torch.cat([self.up2_ps(u1), e1], dim=1)
        u2 = F.relu(self.dec2_fusion(u2))
        u2 = F.relu(self.dec2(u2))

        # --- FINAL OUTPUTS ---
        t_residual = self.t_head_residual(u2)
        
        # Final Transmission = DCP Prior + Learned Residual
        t_final = torch.clamp(t_dcp + t_residual, 0.0, 1.0)
        A_spatial = self.A_head_spatial(u2)

        return t_final, A_spatial

## Stage 2 Code

**PhysConvNeXtBlock**

In [5]:
class PhysConvNeXtBlock(nn.Module):
    """
    ConvNeXt V2 Block: Best for local textures and edges.
    Includes Adaptive Layer Norm (AdaLN) for Time Embedding injection.
    """
    def __init__(self, dim, mult=2):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim) 
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(1e-6 * torch.ones((dim)), requires_grad=True)

    def forward(self, x, t_emb=None):
        inp = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        
        if t_emb is not None:
            x = self.norm(x)
            scale, shift = t_emb.chunk(2, dim=1)
            x = x * (1 + scale.unsqueeze(1).unsqueeze(1)) + shift.unsqueeze(1).unsqueeze(1)
        else:
            x = self.norm(x)
            
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        return inp + x


**PhysBiMambaBlock**

In [6]:
class LocalFeatureExtractor(nn.Module):
    """ 
    Adaptive Parallel Branch.
    Uses Inverted Bottleneck (Expand -> Depthwise -> Project).
    Automatically calculates padding to keep spatial dimensions constant.
    """
    def __init__(self, dim, kernel_size=3, expansion_factor=2, dilation=2):
        super().__init__()
        
        hidden_dim = int(dim * expansion_factor)
        
        # Dynamic Padding Calculation:
        # P = (dilation * (kernel_size - 1)) / 2
        # This ensures the output size equals the input size.
        padding = (dilation * (kernel_size - 1)) // 2
        
        self.net = nn.Sequential(
            # 1. Pointwise Expansion
            nn.Conv2d(dim, hidden_dim, kernel_size=1),
            nn.GELU(),
            
            # 2. Adaptive Depthwise Conv
            nn.Conv2d(hidden_dim, hidden_dim, 
                      kernel_size=kernel_size, 
                      padding=padding, 
                      dilation=dilation,
                      groups=hidden_dim), # Depthwise
            nn.GELU(),
            
            # 3. Pointwise Projection
            nn.Conv2d(hidden_dim, dim, kernel_size=1)
        )

    def forward(self, x):
        return self.net(x)


In [7]:
class PhysBiMambaBlock(nn.Module):
    """
    Bidirectional Mamba Block (BiMamba)
    Scans the image Forward AND Backward so the top-left pixel
    can 'see' the bottom-right pixel.
    """
    def __init__(self, dim, dropout = 0.05):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

        # --- Horizontal Mamba -----
        self.mamba_h_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_h_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)

        # --- Vertical Mamba ---
        self.mamba_v_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        
        # Fuses Fwd+Bwd direction
        self.fusion_linear = nn.Linear(dim * 4, dim)

        self.local_conv = LocalFeatureExtractor(dim, 
                                                kernel_size=3, 
                                                dilation=1)
        
        # Optional: A Gate to let the network choose emphasis
        self.mixer = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )

        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, t_emb=None):
        B, C, H, W = x.shape
        residual = x
        
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)

        if t_emb is not None:
            scale, shift = t_emb.chunk(2, dim=1)
            x_norm = x_norm * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

        # ---------------------------------------------------------
        # 2. HORIZONTAL SCANS (Raster Order)
        # ---------------------------------------------------------
        # Forward ->
        out_h_fwd = self.mamba_h_fwd(x_norm)
        
        # Backward <-
        x_flip = torch.flip(x_norm, dims=[1])
        out_h_bwd = self.mamba_h_bwd(x_flip)
        out_h_bwd = torch.flip(out_h_bwd, dims=[1]) # Flip back

        # ---------------------------------------------------------
        # 3. VERTICAL SCANS (Column-Major Order)
        # ---------------------------------------------------------
        # Reshape to Image -> Transpose (Swap H and W) -> Flatten
        # Result: (B, W*H, C). Now 'neighbors' in seq are vertical neighbors.
        x_v_img = x_norm.view(B, H, W, C).permute(0, 2, 1, 3) 
        x_v_flat = x_v_img.flatten(1, 2)
        
        # Down v
        out_v_fwd = self.mamba_v_fwd(x_v_flat)
        
        # Up ^
        x_v_flip = torch.flip(x_v_flat, dims=[1])
        out_v_bwd = self.mamba_v_bwd(x_v_flip)
        out_v_bwd = torch.flip(out_v_bwd, dims=[1])
        
        # Un-Transpose Vertical Outputs back to Horizontal Order
        # (B, W*H, C) -> (B, W, H, C) -> (B, H, W, C) -> (B, L, C)
        out_v_fwd = out_v_fwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        out_v_bwd = out_v_bwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        
        ## ---------------------------------------------------------
        # 4. Global Fusion
        # ---------------------------------------------------------
        # Combine all 4 views of the image
        global_feat = self.fusion_linear(
            torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
        )

        # ---------------------------------------------------------
        # 5. Local Branch (Conv)
        # ---------------------------------------------------------
        # Reshape for Conv2d
        x_img_norm = x_norm.transpose(1, 2).view(B, C, H, W)
        local_feat = self.local_conv(x_img_norm)
        local_feat = local_feat.flatten(2).transpose(1, 2)

        
        # ---------------------------------------------------------
        # 6. Gated Output
        # ---------------------------------------------------------
        combined = torch.cat([global_feat, local_feat], dim=-1)
        z = self.mixer(combined)
        
        fused = global_feat * z + local_feat * (1 - z)
        
        x_out = self.out_proj(fused)
        
        # Reshape to (B, C, H, W) for residual add
        x_out = x_out.transpose(1, 2).view(B, C, H, W)
        x_out = self.dropout(x_out)
        
        return residual + x_out


In [8]:
class CAGatedFusion(nn.Module):
    """
    Channel Attention Gating (Version 2).
    Decides 'what features' (texture vs fog) to fuse using Global Context.
    """
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),          # Squeeze (Global Context)
            nn.Conv2d(dim * 2, dim // 2, 1),  # Compress
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 2, dim * 2, 1),  # Excite
            nn.Sigmoid()                      # Weight
        )
        self.conv = nn.Conv2d(dim, dim, 1)

    def forward(self, dec_feat, enc_feat):
        combined = torch.cat([dec_feat, enc_feat], dim=1)
        weights = self.attn(combined)
        w_dec, w_enc = weights.chunk(2, dim=1)
        # Channel-wise weighted fusion
        fused = (dec_feat * w_dec) + (enc_feat * w_enc)
        return self.conv(fused)


In [9]:
class PixelShuffleUpsample(nn.Module):
    """
    SOTA Trick: Replaces ConvTranspose2d to eliminate checkerboard artifacts.
    """
    def __init__(self, dim_in, dim_out):
        super().__init__()
        # We need to project to (dim_out * 4) so PixelShuffle(2) results in dim_out
        self.conv = nn.Conv2d(dim_in, dim_out * 4, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(2) # Scale x2
        
    def forward(self, x):
        return self.pixel_shuffle(self.conv(x))


## Flow Matching UNet

In [10]:
class Stage2_FlowMatching_UNet(nn.Module):
    def __init__(self, 
                 in_channels = 10,     # x_t (3) + hazy_img (3) + t_map (1) + A (3)
                 out_channels = 3,     # predicted vector field v (3)
                 base_dim = 64, 
                 dim_mults = [1, 2, 4, 8],
                 enc_blocks = [2, 2, 4], 
                 dec_blocks = [4, 2, 2],
                 num_mid_blocks = 4,
                 physics_guided=True):
        super().__init__()

        self.dims = [base_dim * m for m in dim_mults]
        self.physics_guided = physics_guided

        # --- Time & Physics Embedding (Global Conditioning) --- 
        time_dim = base_dim * 4
        self.time_mlp = nn.Sequential(
            nn.Linear(base_dim, time_dim), 
            nn.SiLU(), 
            nn.Linear(time_dim, time_dim)
        )

        # Fuses the Flow Timestep with the Frozen Atmospheric Light (A)
        self.phys_gate = nn.Sequential(
            nn.Linear(time_dim + 3, time_dim), 
            nn.SiLU(), 
            nn.Linear(time_dim, time_dim)
        )

        # --- ENCODER ---
        self.init_conv = nn.Conv2d(in_channels, self.dims[0], 3, 1, 1)
        self.down_time_projs = nn.ModuleList()
        self.up_time_projs = nn.ModuleList()

        for i in range(len(self.dims) - 1):
            dim_in, dim_out = self.dims[i], self.dims[i + 1]
            self.down_time_projs.append(nn.Linear(time_dim, dim_in * 2))

            blocks = nn.ModuleList([PhysConvNeXtBlock(dim_in)])
            num_mamba = enc_blocks[i] if i < len(enc_blocks) else 1
            for _ in range(num_mamba):
                blocks.append(PhysBiMambaBlock(dim_in))

            self.downs.append(blocks)
            self.downsamples.append(nn.Conv2d(dim_in, dim_out, 4, 2, 1))

        # --- BOTTLENECK ---
        mid_dim = self.dims[-1]
        self.mid_time_proj = nn.Linear(time_dim, mid_dim * 2)
        self.mid_blocks = nn.ModuleList()
        for _ in range(num_mid_blocks):
            self.mid_blocks.append(PhysBiMambaBlock(mid_dim))

        # --- DECODER ---
        self.ups = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        self.gates = nn.ModuleList()

        for idx, i in enumerate(range(len(self.dims) - 2, -1, -1)):
            dim_in, dim_out = self.dims[i+1], self.dims[i]
            self.up_time_projs.append(nn.Linear(time_dim, dim_out * 2))

            self.up_samples.append(PixelShuffleUpsample(dim_in, dim_out))
            self.gates.append(CAGatedFusion(dim_out))

            layers = nn.ModuleList()
            num_mamba = dec_blocks[idx] if idx < len(dec_blocks) else 1
            for _ in range(num_mamba):
                if i > 0: layers.append(PhysBiMambaBlock(dim_out))
                else: layers.append(PhysConvNeXtBlock(dim_out))

            layers.append(PhysConvNeXtBlock(dim_out)) 
            self.ups.append(layers)

        # Output Projection for the vector field
        self.final_conv = nn.Conv2d(self.dims[0], out_channels, 1)

    def get_sinusoidal_emb(self, t, device):
        half_dim = self.dims[0] // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

    def forward(self, x_t, timestep, hazy_img, t_map, A):
        """
        Args:
            x_t: Noisy image at current timestep (B, 3, H, W)
            timestep: Flow matching timestep (B,)
            hazy_img: Original hazy input (B, 3, H, W)
            t_map: Stage 1 Transmission Map (B, 1, H, W)
            A: Stage 1 Atmospheric Light (B, 3)
        """
        # 1. Global Time & Physics Embedding 
        t_emb_raw = self.get_sinusoidal_emb(timestep, x_t.device) 
        t_vec = self.time_mlp(t_emb_raw)

        # Inject global atmospheric light 'A' into the flow timeline
        A_global = F.adaptive_avg_pool2d(A, 1).view(-1, 3)
        phys_cond = torch.cat([t_vec, A_global], dim = -1)
        t_vec = self.phys_gate(phys_cond)

        # 2. Input Setup (10 Channels total)
        # Spatial 'A' goes directly into the convolutions alongside the noise
        x = torch.cat([x_t, hazy_img, t_map, A], dim = 1)
        h = self.init_conv(x)
        skips = []

        # 3. ENCODER 
        for i, (block_list, down_layer) in enumerate(zip(self.downs, self.downsamples)):
            t_emb = self.down_time_projs[i](t_vec)
            for layer in block_list:
                h = layer(h, t_emb)
            skips.append(h)
            h = down_layer(h)

        # 4. BOTTLENECK
        t_emb_mid = self.mid_time_proj(t_vec)
        for block in self.mid_blocks:
            h = block(h, t_emb_mid)

        # 5. DECODER
        for i in range(len(self.ups)):
            h = self.up_samples[i](h)
            if len(skips) > 0:
                skip = skips.pop()
                h = self.gates[i](h, skip)

            block_list = self.ups[i]
            t_emb_dec = self.up_time_projs[i](t_vec)

            for layer in block_list:
                h = layer(h, t_emb_dec)

        # 6. Physics Guidance & Output
        if self.physics_guided:
            # Guide the final features using the transmission map
            # This mathematically enhances features in clear areas and forces
            # the network to hallucinate more in heavily occluded areas.
            h = h * (1.0 + t_map)

        v_pred = self.final_conv(h)

        return v_pred

### Loss Functions

In the old loss components I will drop: 
- Physics Consistency `loss_phs`:
  - Reconstructs the image using $I = J \cdot t + A(1-t)$ and compares it to the input haze.
  - DROP: Since this is the Stage 2 which receives all of the physics information from the Stage. If we enforce the Flow model to perfectly obey Stage 1's physics, we will losse its generative ability to hallucinate the missing textures.
- VGG Perceptual (`loss_percep`) & Contrastive (`loss_cr`):
  - Both extract VGG features. `loss_precep` pulls the prediction to the clean image. `loss_cr` pulls to the cleann image and pushes away from the hazy image.
  - I will keep the `loss_cr` since it already does the job of Perceptual loss in its positive term.
- FFT Frequency Loss `loss_fft`:
  -  Forces the high-frequency edges and Fourier phase of the prediction to match the clean image.
  -  DROP: Flow matching starts from 100% random noise. If you penalize the network for not having perfect sharp edges at Timestep 0.9 (when the image is just a cloud of static), the gradients will explode and clash with the Flow velocity.
- SSIM Loss `loss_ssim`:
    -  Penalizes structural differences.
    -  DROP. SSIM is a terrible loss function for generative models. It is highly sensitive to tiny pixel shifts. To minimize SSIM loss, generative networks will naturally output blurry, smooth pixels instead of sharp, realistic textures.
- TV Loss `loss_tv` & ATM Regularizer `loss_atm`
    - What they do: Try to smooth the transmission map and keep $A$ above 0.05.
    - INVALID (DELETE). In our new pipeline, Stage 1 is frozen. You cannot compute gradients for $t$ and $A$ during Stage 2. If you leave these in, PyTorch will throw an error or waste memory.

#### Problem with Gradient Conflict here

Even without noise, if you apply VGG, Contrastive, and FFT losses at the wrong timesteps, your model will break. In this I2I paradigm, the conflict is not caused by "noise," but by vanishing math and exploding gradients.To apply image losses (like VGG or SSIM), we have to guess what the final Clear Image ($J_{pred}$) will look like based on the model's current velocity. The formula to project forward to $t=1$ is:$$J_{pred} = x_t + (1 - t) \cdot v_{pred}$$

Here is what happens at the two extremes of time:

**Case 1: When $t \approx 0.99$ (Almost the Clear Image)**
- What $x_t$ looks like: It looks 99% like the perfect ground-truth clean image.
- The Math Problem: Look at the multiplier: $(1 - 0.99) = 0.01$. So, $J_{pred} = x_{0.99} + 0.01 \cdot v_{pred}$.
- The Conflict (Exploding Gradients): Imagine your VGG loss looks at $J_{pred}$ and says, "This pixel is slightly too dark, make it brighter by 1 unit." Because $v_{pred}$ is multiplied by $0.01$, the network has to force $v_{pred}$ to change by 100 units just to satisfy that tiny VGG request!
- The Result: Your gradients explode to infinity. The vector field $v$ goes completely chaotic. To stop the gradients from exploding, the optimizer panics and forces $v=0$ to minimize the damage.

**Case 2: When $t \approx 0.01$ (Almost the Hazy Image)**
- What $x_t$ looks like: It looks exactly like the input hazy image. It is blurry, low-contrast, and lacks detail.
- The Math Problem: The multiplier is $(1 - 0.01) = 0.99$. So, $J_{pred} \approx x_{0.01} + 1.0 \cdot v_{pred}$.
- The Conflict (The "One-Shot" Burden): If you apply FFT (Frequency) or VGG loss here, you are demanding that the network's velocity ($v_{pred}$) perfectly contains every single high-frequency edge and texture needed to fix the image in one massive leap. Flow matching is designed to be a smooth, incremental journey. If you force the model to solve the whole image perfectly at $t=0.01$, it behaves like a standard, often-blurry ResNet, completely defeating the purpose of the Flow formulation.

**3. The "Goldilocks Zone" for Your Losses**

Because you are using Direct Image-to-Image Flow, your standard Flow MSE Loss ($||v_{pred} - u_t||^2$) is perfectly safe and stable at all timesteps.

However, your "heavy" image losses (Contrastive, VGG, FFT) are only safe in the middle of the trajectory.
- They shouldn't be applied near $t=1$ because of exploding gradients.
- They shouldn't be applied near $t=0$ because it forces the model to do too much work at once.

In [15]:
import lpips 

class LPIPS_ContrastiveLoss(nn.Module):
    """
    Combines the Push/Pull dynamics of Contrastive Learning 
    with the human-calibrated perceptual features of LPIPS.
    """
    def __init__(self, net='vgg'):
        super().__init__()
        # Load the official LPIPS model (VGG backbone is standard)
        # It automatically handles the VGG freezing and layer extraction internally
        self.lpips_model = lpips.LPIPS(net=net).eval()
        
        # Ensure it requires no gradients (frozen critic)
        for param in self.lpips_model.parameters():
            param.requires_grad = False

    def forward(self, restored, clear, hazy):
        """
        Inputs should preferably be in range [-1, 1].
        If they are in [0, 1], we automatically scale them.
        """
        # 1. Normalize to [-1, 1] as expected by official LPIPS
        if restored.min() >= 0.0 and restored.max() <= 1.0:
            restored = restored * 2.0 - 1.0
            clear = clear * 2.0 - 1.0
            hazy = hazy * 2.0 - 1.0

        # SAFETY CLAMP
        # Crucial for Flow Matching because J_pred is a projection 
        # that can easily overshoot valid pixel ranges.
        restored = torch.clamp(restored, -1.0, 1.0)
        clear = torch.clamp(clear, -1.0, 1.0)
        hazy = torch.clamp(hazy, -1.0, 1.0)

        # 2. Positive Distance (Pull toward Clear)
        # lpips returns shape (B, 1, 1, 1), so we take the mean
        d_pos = self.lpips_model(restored, clear).mean()

        # 3. Negative Distance (Push away from Hazy)
        # We don't need to track gradients for the negative target itself
        with torch.no_grad():
            d_neg = self.lpips_model(restored, hazy).mean()

        # 4. Contrastive Formula: Minimize D_pos / (D_neg + epsilon)
        # As D_pos gets smaller, loss goes down.
        # As D_neg gets larger (further from haze), loss goes down.
        loss = d_pos / (d_neg + 1e-7)

        return loss

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from data.utils import restandardize_tensor
from losses import PerceptualLoss, ContrastiveLoss

class Stage2_FlowMatchingLoss_Lean(nn.Module):
    def __init__(self, loss_config=None):
        super().__init__()
        self.mse_none = nn.MSELoss(reduction='none') 
        
        # We keep only Contrastive (which relies on VGG inside it)
        self.perceptual_vgg = PerceptualLoss() 
        self.contrastive = LPIPS_ContrastiveLoss(net='vgg')

        self.weights = {
            "w_flow": 1.0,        
            "w_cr": 0.1,         # Replaces Perceptual entirely
            "density_boost": 5.0 
        }

    def forward(self, pred_v, target_v, x_t, timestep, clean_img, hazy_img,
                frozen_t_map, current_epoch=None, total_epochs=100):
        
        # --- A. TIME NORM ---
        t_norm = timestep.float() / 1000.0 if timestep.max() > 1.0 else timestep.float()
        t_expand = t_norm.view(-1, 1, 1, 1)

        # --- B. DENSITY-AWARE FLOW LOSS (The Core Engine) ---
        raw_v_loss = self.mse_none(pred_v, target_v)
        
        # Adaptive boosting over epochs
        max_boost = self.weights["density_boost"]
        current_boost = max_boost if current_epoch is None else \
                        (max(0.0, min(current_epoch / float(total_epochs), 1.0)) ** 2) * max_boost 
        
        # frozen_t_map directs the network to focus on thick haze
        t_guide = frozen_t_map.detach().mean(dim=1, keepdim=True)
        pixel_weight = 1.0 + current_boost * (1.0 - t_guide)
        
        loss_v = (raw_v_loss * pixel_weight).mean()
        total_loss = self.weights["w_flow"] * loss_v
        
        loss_cr_val = 0.0

        # --- C. TIMESTEP GATING (THE GOLDILOCKS ZONE) ---
        # 1. We avoid t < 0.2 (Near Hazy) because forcing one-shot generation breaks the flow engine.
        # 2. We avoid t > 0.8 (Near Clean) because dividing by (1-t) causes exploding gradients.
        valid_mask = ((t_norm > 0.2) & (t_norm < 0.8)).view(-1)
        
        if valid_mask.any():
            # Filter out the batch items that fall outside the safe zone
            valid_pred_v = pred_v[valid_mask]
            valid_xt = x_t[valid_mask]
            valid_t_expand = t_expand[valid_mask]
            
            # Predict what the clean image J looks like right now
            # Math: x_1 = x_t + (1 - t) * v_pred
            J_pred_raw = valid_xt + (1 - valid_t_expand) * valid_pred_v
            J_pred_safe = torch.clamp(restandardize_tensor(J_pred_raw), 0.0, 1.0)

            # We need to restandardize them
            clean_safe = restandardize_tensor(clean_img[valid_mask])
            hazy_safe = restandardize_tensor(hazy_img[valid_mask])

            # Apply Contrastive Loss ONLY to these valid samples
            loss_cr_batch = self.contrastive(J_pred_safe, clean_safe, hazy_safe)
            
            total_loss += (self.weights["w_cr"] * loss_cr_batch)
            loss_cr_val = loss_cr_batch.item()

        return total_loss, {
            "Total": total_loss.item(),
            "Flow_MSE": loss_v.item(),
            "Contrastive": loss_cr_val
        }

### Generate Synthetic Haze from the MCBM

Markov Chain Brownian Motion

In [18]:
import numpy as np
from PIL import Image
import cv2 
import os 
import random 

from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt

In [ ]:
def generate_mcbm_haze(rows=256, cols=256, iterations_factor=5, sigma=25):
    """
    Generates a non-homogeneous haze density map using MCBM.
    
    Args:
        rows, cols: Dimensions of the density map.
        iterations_factor: Multiplier for total steps (n = rows * cols * factor).
        sigma: Standard deviation for Brownian motion and Gaussian smoothing.
    """
    # 1. Initialize the 2D density array (Z)
    Z = np.zeros((rows, cols))

    # 2. Define Markov Chain transition probabilities (Up, Down, Left, Right)
    # [cite: 127]
    moves = [(0, 1), (0, -1), (1, 0), (-1, 0)]

    # 3. Select a random starting pixel [cite: 125]
    curr_r, curr_c = np.random.randint(0, rows), np.random.randint(0, cols)
    Z[curr_r, curr_c] += 1

    # 4. Simulate Markov Chain with Brownian Motion [cite: 123, 131]
    num_steps = rows * cols * iterations_factor
    for _ in range(num_steps):
        # Step A: Markov Chain Step (Random move to neighbor) [cite: 127]
        dr, dc = random.choice(moves)
        curr_r = np.clip(curr_r + dr, 0, rows - 1)
        curr_c = np.clip(curr_c + dc, 0, cols - 1)
        
        # Step B: Brownian Motion (Random displacement from normal distribution) [cite: 129]
        # Delta i, Delta j ~ N(0, sigma^2)
        delta_r = int(round(np.random.normal(0, sigma)))
        delta_c = int(round(np.random.normal(0, sigma)))
        
        # Calculate target coordinates with periodic boundary (wrapping) or clipping
        target_r = (curr_r + delta_r) % rows
        target_c = (curr_c + delta_c) % cols
        
        # Update density map [cite: 129]
        Z[target_r, target_c] += 1

    # 5. Post-Processing: Smoothing and Normalization 
    # Apply Gaussian filter for realistic appearance
    Z_smoothed = gaussian_filter(Z, sigma=sigma/5.0) 
    
    # Normalize to [0, 1] range
    Z_min, Z_max = np.min(Z_smoothed), np.max(Z_smoothed)
    Z_norm = (Z_smoothed - Z_min) / (Z_max - Z_min)
    
    return Z, Z_norm